In [ ]:
# Geopandas for geodataframe to manipulate GIS coordinates
# Pandas (as pandas to distinguish from GPD) for additional methods
# SQLAlchemy, and URL module to create engine
# Getpass for secure 

import geopandas as gpd
import pandas
import plotly.express as px
# If map fails to render, may need to:
# from IPython.display import HTML

In [ ]:
# Read the parquet file into the dataframe. You can find the clean data in the dataset on Kaggle:
# https://www.kaggle.com/datasets/mrdragonbishop/cyclistic-202505-202605
capstone_data_raw = gpd.read_parquet("cleaned_data.parquet")

# Remove all rides in the top and bottom percentiles of duration. At the lower end, this removes
# docking errors, immediate re-docks, accidental unlocks, etc., and ensures a balanced picture of
# the customers Cyclistic is targeting.

capstone_data = capstone_data_raw[
    (capstone_data_raw["ride_duration"] >= capstone_data_raw["ride_duration"].quantile(0.01)) &    
    (capstone_data_raw["ride_duration"] <= capstone_data_raw["ride_duration"].quantile(0.99)) 
]

# Add columns that list the day of week, and month, of each ride
capstone_data['day_of_week'] = capstone_data['start_time'].dt.day_name()
capstone_data['month'] = capstone_data['start_time'].dt.month

# Order the days of the week for correct sorting for analysis
days_order = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
capstone_data['day_of_week'] = pandas.Categorical(
    capstone_data['day_of_week'], categories=days_order, ordered=True
    )
# Order the months for correct sorting for analysis
capstone_data['month'] = capstone_data['start_time'].dt.month_name() 
months_order = ["January", "February", "March", "April", "May", "June",
                "July", "August", "September", "October", "November", "December"]
capstone_data['month'] = pandas.Categorical(
    capstone_data['month'], categories=months_order
)
# display descriptive attributes of dataframe for auditing 
print("See list of columns and data types:")
capstone_data.info()
capstone_data.head()

See list of columns and data types:
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 3800822 entries, 0 to 3878388
Data columns (total 12 columns):
 #   Column             Dtype          
---  ------             -----          
 0   ride_id            str            
 1   member_type        str            
 2   bike_type          str            
 3   start_time         datetime64[us] 
 4   end_time           datetime64[us] 
 5   ride_duration      timedelta64[us]
 6   start_location     str            
 7   start_coordinates  geometry       
 8   end_location       str            
 9   end_coordinates    geometry       
 10  day_of_week        category       
 11  month              category       
dtypes: category(2), datetime64[us](2), geometry(2), str(5), timedelta64[us](1)
memory usage: 624.8 MB


,ride_id,member_type,bike_type,start_time,end_time,ride_duration,start_location,start_coordinates,end_location,end_coordinates,day_of_week,month
0,79DA749467DEE3CF,member,electric_bike,2025-11-06 06:15:26.757,2025-11-06 06:21:53.889,0 days 00:06:27.132000,Michigan Ave & 14th St,POINT (-87.62373 41.86406),Clark St & Ida B Wells Dr,POINT (-87.63058 41.87593),Thursday,November
1,79DA77741F29F6E1,casual,electric_bike,2026-04-10 18:29:51.091,2026-04-10 18:40:46.486,0 days 00:10:55.395000,Kingsbury St & Kinzie St 2,POINT (-87.63839 41.88919),The Salt Shed,POINT (-87.65978 41.90689),Friday,April
2,79DA7D0ED4ACC5FF,casual,electric_bike,2026-04-12 10:23:57.796,2026-04-12 10:29:08.142,0 days 00:05:10.346000,Green St & Lake St,POINT (-87.64841 41.88558),Kingsbury St & Kinzie St 1,POINT (-87.63835 41.88907),Sunday,April
3,79DA7E50986141F3,member,classic_bike,2025-06-23 17:08:49.038,2025-06-23 17:19:42.404,0 days 00:10:53.366000,Sedgwick St & Huron St,POINT (-87.63844 41.89466),Canal St & Adams St,POINT (-87.6399 41.87925),Monday,June
4,79DA7F6C897BB395,member,classic_bike,2026-05-01 23:09:51.553,2026-05-01 23:19:32.775,0 days 00:09:41.222000,Sheridan Rd & Irving Park Rd,POINT (-87.6544 41.95425),Sheridan Rd & Lawrence Ave,POINT (-87.65469 41.96952),Friday,May


# Data Analysis

## Methodology

This analysis employs descriptive statistical methods and geospatial analysis to identify differences in bike-share usage between annual members and casual riders. It compares rider behavior across temporal, geographic, and quantitative dimensions to understand use patterns.

Ride counts and ride durations were aggregated by member type and analyzed across day_of_week, month, bike_type, and neighborhood to identify trends in ridership volume. Measures of central tendency, such as mean and median, distinguished trip characteristics of rider groups. Standard deviation helped evaluate variability within subgroups, and minimum and maximum values were used to identify potential outliers and assess the quality and integrity of the data.

In [4]:
# Compare members and casual users
display("Ride length statistics grouped by member_type:", capstone_data.groupby('member_type')['ride_duration'].agg(['mean', 'median', 'max', 'min']))

'Ride length statistics grouped by member_type:'

,mean,median,max,min
member_type,,,,
casual,0 days 00:18:02.306868,0 days 00:12:27.531000,0 days 01:35:35.536000,0 days 00:01:00.611000
member,0 days 00:11:45.671733,0 days 00:08:46.423000,0 days 01:35:35.380000,0 days 00:01:00.604000


In [ ]:
# Analyze ridership data by membership type and weekday
member_weekday = capstone_data.groupby(['member_type', 'day_of_week']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_duration', 'mean')
).reset_index()
print("Member Ridership Statistics By Weekday\n") 
display(member_weekday)

# Analyze ridership data by bike type and weekday
rider_weekday = capstone_data.groupby(['bike_type', 'day_of_week']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_duration', 'mean')
).reset_index()
print("Bike_type Ridership Statistics By Weekday\n") 
display(rider_weekday)

Member Ridership Statistics By Weekday



,member_type,day_of_week,number_of_rides,average_duration
0,casual,Sunday,230381,0 days 00:20:21.385071
1,casual,Monday,158004,0 days 00:18:03.694200
2,casual,Tuesday,148315,0 days 00:16:00.715742
3,casual,Wednesday,148091,0 days 00:15:23.987195
4,casual,Thursday,166300,0 days 00:16:03.564103
5,casual,Friday,204163,0 days 00:17:36.104259
6,casual,Saturday,273622,0 days 00:20:07.720217
7,member,Sunday,269315,0 days 00:12:58.403210
8,member,Monday,360063,0 days 00:11:28.385211
9,member,Tuesday,398782,0 days 00:11:19.794998


Bike_type Ridership Statistics By Weekday



,bike_type,day_of_week,number_of_rides,average_duration
0,classic_bike,Sunday,266337,0 days 00:18:13.306240
1,classic_bike,Monday,263310,0 days 00:15:06.385824
2,classic_bike,Tuesday,270836,0 days 00:14:06.133721
3,classic_bike,Wednesday,265485,0 days 00:13:50.702688
4,classic_bike,Thursday,272515,0 days 00:14:15.015225
5,classic_bike,Friday,271890,0 days 00:15:23.075579
6,classic_bike,Saturday,298148,0 days 00:18:17.167455
7,electric_bike,Sunday,233359,0 days 00:14:16.327248
8,electric_bike,Monday,254757,0 days 00:11:48.242015
9,electric_bike,Tuesday,276261,0 days 00:11:07.539362


In [6]:
# Analyze ridership data by membership type and month
member_month = capstone_data.groupby(['member_type', 'month']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_duration', 'mean')
).reset_index()
print("Member Ridership Statistics By Month\n") 
display(member_month)

# Analyze ridership data by bike type and month
rider_month = capstone_data.groupby(['bike_type', 'month']).agg(
    number_of_rides=('ride_id', 'count'),
    average_duration=('ride_duration', 'mean')
).reset_index()
print("Bike_type Ridership Statistics By Month\n") 
display(rider_month)

Member Ridership Statistics By Month



,member_type,month,number_of_rides,average_duration
0,casual,January,16195,0 days 00:10:19.817092
1,casual,February,28977,0 days 00:13:07.702984
2,casual,March,58855,0 days 00:15:48.237323
3,casual,April,85132,0 days 00:16:00.649981
4,casual,May,153117,0 days 00:18:05.959939
5,casual,June,185828,0 days 00:19:50.696562
6,casual,July,200712,0 days 00:19:34.699687
7,casual,August,212812,0 days 00:19:58.891845
8,casual,September,166688,0 days 00:18:15.426467
9,casual,October,139291,0 days 00:16:49.316241


Bike_type Ridership Statistics By Month



,bike_type,month,number_of_rides,average_duration
0,classic_bike,January,43205,0 days 00:10:48.953157
1,classic_bike,February,66255,0 days 00:12:01.869145
2,classic_bike,March,94819,0 days 00:13:52.659034
3,classic_bike,April,131543,0 days 00:14:14.328470
4,classic_bike,May,192024,0 days 00:16:09.192887
5,classic_bike,June,248051,0 days 00:17:24.327398
6,classic_bike,July,258805,0 days 00:17:10.622253
7,classic_bike,August,274494,0 days 00:17:25.725586
8,classic_bike,September,234934,0 days 00:15:56.580194
9,classic_bike,October,206210,0 days 00:14:42.822107


In [46]:
# Let's expand on our descriptive analysis:

# Define the statistical metrics for stakeholders:
metrics = ['mean', 'median', 'min', 'max', 'std']

# A supplementary list of advanced metrics for a more technical audience:
# from scipy.stats import variation
# advanced_metrics = ['sem', 'skew', 'kurt', variation]

# The first key data element is ride duration, which needs to be converted to numeric minutes for legibility. 
# The second is the number of rides, which requires grouping and aggregation.
capstone_data['ride_duration_min'] = capstone_data['ride_duration'].dt.total_seconds() / 60

# Count the no. of rides that occurred on each specific day, grouping by date and day of week.
count_stats_weekly = (
    capstone_data
    .groupby(['member_type', 'day_of_week', capstone_data['start_time'].dt.date])
    ['ride_id']
    .count()
    .rename('number_of_rides')
    .reset_index()
    .groupby(['member_type', 'day_of_week'])['number_of_rides']
    .agg(metrics)
    .add_prefix('count_')
    .reset_index()
)
# Group the duration of rides by member type, month, then day_of_week
duration_stats_weekly = (
    capstone_data
    .groupby(['member_type', 'day_of_week'])['ride_duration_min']
    .agg(metrics)
    .add_prefix('duration_')
    .reset_index()
)
# Joining the two tables while resetting the index from multilevel to a numeric range
weekly_stats = count_stats_weekly.merge(
    duration_stats_weekly,
    on=['member_type', 'day_of_week'],
    how='outer'
)
#Reordering the days after aggregation
weekly_stats['day_of_week'] = pandas.Categorical(
    weekly_stats['day_of_week'],
    categories=days_order,
    ordered=True
)

display(weekly_stats.round(2))

# Count the no. of rides that occurred on each specific day, grouping by date and month.
count_stats_monthly = (
    capstone_data
    .groupby(['member_type', 'month', capstone_data['start_time'].dt.date])
    ['ride_id']
    .count()
    .rename('number_of_rides')
    .reset_index()
    .groupby(['member_type', 'month'])['number_of_rides']
    .agg(metrics)
    .add_prefix('count_')
    .reset_index()   
)
# Group the duration of rides by member type, month, then day_of_week
duration_stats_monthly = (
    capstone_data
    .groupby(['member_type', 'month'])['ride_duration_min']
    .agg(metrics)
    .add_prefix('duration_')
    .reset_index()
)
# Joining the two tables while resetting the index from multilevel to a numeric range
monthly_stats = count_stats_monthly.merge(
    duration_stats_monthly,
    on=['member_type', 'month'],
    how='outer'
)

#Reordering the months after aggregation
monthly_stats['month'] = pandas.Categorical(
    monthly_stats['month'],
    categories=months_order,
    ordered=True
)

display(monthly_stats.round(2))

,member_type,day_of_week,count_mean,count_median,count_min,count_max,count_std,duration_mean,duration_median,duration_min,duration_max,duration_std
0,casual,Sunday,4346.81,3722.0,47,11068,3229.98,20.36,14.47,1.01,95.59,17.76
1,casual,Monday,3038.54,3062.5,105,9278,2253.70,18.06,12.11,1.01,95.59,16.95
2,casual,Tuesday,2852.21,2622.0,208,6746,1924.78,16.01,10.87,1.01,95.59,15.28
3,casual,Wednesday,2847.90,2632.0,267,7106,1938.12,15.40,10.58,1.01,95.57,14.55
4,casual,Thursday,3198.08,2814.0,314,7715,2154.57,16.06,11.00,1.01,95.58,15.08
5,casual,Friday,3926.21,3959.5,83,8995,2619.61,17.60,12.18,1.01,95.58,16.00
6,casual,Saturday,5162.68,4529.0,58,11367,3826.25,20.13,14.51,1.01,95.59,17.30
7,member,Sunday,5081.42,5687.0,247,8359,2595.67,12.97,9.44,1.01,95.54,11.30
8,member,Monday,6924.29,8121.5,925,10534,2916.55,11.47,8.51,1.01,95.59,9.73
9,member,Tuesday,7668.88,8282.5,1690,11901,2982.44,11.33,8.58,1.01,95.59,9.26


,member_type,month,count_mean,count_median,count_min,count_max,count_std,duration_mean,duration_median,duration_min,duration_max,duration_std
0,casual,January,522.42,465.0,47,1333,355.41,10.33,7.36,1.01,93.15,9.80
1,casual,February,1034.89,834.5,306,2770,692.20,13.13,8.63,1.01,95.22,13.32
2,casual,March,1898.55,1519.0,390,7051,1385.45,15.80,10.57,1.01,95.46,15.20
3,casual,April,2837.73,2648.5,1197,5220,1112.85,16.01,10.77,1.01,95.58,15.30
4,casual,May,4784.91,4424.0,58,9569,2155.03,18.10,12.52,1.01,95.58,16.49
5,casual,June,6194.27,5919.0,1835,11233,2101.29,19.84,14.20,1.01,95.59,17.16
6,casual,July,6474.58,5843.0,3811,11367,1774.34,19.58,13.77,1.01,95.59,17.29
7,casual,August,6864.90,6228.0,2687,11225,2233.75,19.98,14.24,1.01,95.59,17.39
8,casual,September,5556.27,5039.5,3302,9885,1574.26,18.26,12.69,1.01,95.59,16.51
9,casual,October,4493.26,4255.0,2116,9532,2053.79,16.82,11.52,1.01,95.59,15.63


In [ ]:
# read neighborhoods information as raw string literal (thanks python interpreter!) and then copy to separate dataframe
neighborhoods_raw = gpd.read_file(
    r"github\cyclistic_case_study\data\cleaned_data\chicago_community_areas\geo_export_4fe93f43-fd2e-4afd-a835-4315dcdca354.shp"
)

neighborhoods = neighborhoods_raw.copy()

# Create mapped_data dataframe through left spatial join to retain all left df records
# Join ride to neighborhood if coordinate falls within its polygon
mapped_data = gpd.sjoin(
    capstone_data, neighborhoods, how='left', predicate="within")

# Group rides by neighborhood, membership_type, and bike_type simultaneously.
# Count the number of rides in each combination of the three criteria.
# Unstack three-level index, rotating rider type and biker type up,
# Creating columns whose names are a tuple of strings.
ride_counts = (
    mapped_data.groupby(["area_numbe", "member_type", "bike_type"])
    .size()
    .unstack(["member_type", "bike_type"], fill_value=0)
    )

# Combine each tuple into a column header string joined by "_".
ride_counts.columns = ["_".join(col) for col in ride_counts.columns]

#Reset the index of the dataframe to a series of integers
ride_counts = ride_counts.reset_index()

# NOTE: A small % of rides start or end outside Chicago, so start and end coordinates
# Will return a different set of rides (about .5% from original clean dataset).
# This merge is where the sets can diverge.
neighborhoods = neighborhoods.merge(
    ride_counts, on="area_numbe", how="left"
    )

# Create a list of column names that starts with either member_ or casual_
# replace NaNs with 0s to avoid throwing errors
members_cols = [
    col for col in neighborhoods.columns
    if col.startswith("member_") or col.startswith("casual_")
]

# In the lisf of lists whose names are in the list member_columns, fill NaNs with 0s.
neighborhoods[members_cols] = neighborhoods[members_cols].fillna(0)

# The casual_count column equals the number of records containing "casual" membership type per neighborhood
neighborhoods["casual_count"] = neighborhoods[
    [col for col in neighborhoods.columns if "casual" in col]
    ].sum(axis=1)

# The member_count column equals the number of records containing "casual" membership type per neighborhood
neighborhoods["member_count"] = neighborhoods[
    [col for col in neighborhoods.columns if "member" in col]
    ].sum(axis=1)

# The total_count column equals the sums of the casual_count and member_count columns.
neighborhoods["total_count"] = neighborhoods[
    ['casual_count', 'member_count']
    ].sum(axis=1)

# Create a dataframe without the geographic fields to more easily inspect the differences between
# rider profiles
top_neighborhoods = neighborhoods[
    ["community", "casual_classic_bike", "casual_electric_bike", "member_classic_bike", "member_electric_bike", "casual_count", "member_count", "total_count"]
    ].copy(
    ).sort_values(by="total_count",ascending=False,axis=0)

display(top_neighborhoods)

,community,casual_classic_bike,casual_electric_bike,member_classic_bike,member_electric_bike,casual_count,member_count,total_count
7,NEAR NORTH SIDE,139058,137875,246672,243638,276933,490310,767243
31,LOOP,104744,102915,156222,139729,207659,295951,503610
27,NEAR WEST SIDE,47977,77762,170567,191787,125739,362354,488093
6,LINCOLN PARK,82297,78085,149602,125222,160382,274824,435206
5,LAKE VIEW,55524,68052,132904,113990,123576,246894,370470
...,...,...,...,...,...,...,...,...
63,CLEARING,0,41,0,74,41,74,115
8,EDISON PARK,0,29,0,25,29,25,54
35,OAKLAND,0,9,0,28,9,28,37
53,RIVERDALE,12,17,1,4,29,5,34


## Findings

1. **Ride Duration Analysis** - We investigated the distribution of ride durations, examining the median and average lengths of ride times, and the shape of their distribution. We found that casual riders vary more widely in the duration of their trips, and their ride_durations skew longer, with mean significantly higher than median. This suggests that they tend in comparison to be more likely to be recreational cyclists.
2. **Ride Frequency Analysis** - We aggregated the number of rides across the membership categories, and also aggregated their number by day of week, month, and Chicago neighborhoods. We found that generally speaking, member rides outnumber casual riders, but some areas (such as the Loop) show a higher relative proportion of casual riders, indicating a tendency towards recreational purposes.
3. **Day-of-Week, Monthly Trends** - When ride duration and ride frequency are compared across the days of the week, it is clear that member use peaks early in the week, remains consistent throughout, and then dips at the end of the week, remaining low through the weekend. By contrast, casual riders peak during the weekend and wane during the week.

## Visualization

 ### Chloropleth Maps
 
 By using publicly available shapefiles, we can map key statistical information about ride frequency and ride durations onto the neighborhoods of Chicago. Plotly Express allows for zooming, interaction, and a hovering tooltip.

In [ ]:
map_casual = neighborhoods.explore(
    width=800, height=500,
    column="casual_count",
)
display(map_casual)


In [ ]:
map_member = neighborhoods.explore(
    width=800, height=500,
    column="member_count",
)
display(map_member)

### Violin Plot

Similar to a box and whisker plot, violin plots allow for the communication of both summary statistics and the distribution of data in ways that box plots or bar charts struggle to. Mean and median can describe the typical ride, but do not reveal how rides are distributed around these values. Violin plots show the median and quartiles, the density of observations, how the data skews, and if there are differences and variability between groups.

In [ ]:
capstone_sample = capstone_data.sample(n=50000)
capstone_violin = px.violin(capstone_sample, x='member_type', y='ride_duration_min', 
                            title="Distribution of Rides Over Time by Member Type", width=800, height=450)
capstone_violin.show()

### Line Plots

We will need to analyze the duration and number of rides using mean, total, and categorization by day_of_week and month. A line plot allows for clear contrast for shifts in data points over time.

In [64]:
# weekly_line = sns.lineplot(data=weekly_stats, x='day_of_week', y='duration_mean', hue="member_type")
line_monthly_duration = px.line(data_frame=monthly_stats, x='month', y="duration_mean", color="member_type",
                                 title="Monthly Average Ride Duration by Member Type",width=700, height=400)
line_monthly_duration.show()

In [57]:
# sns.lineplot(data=monthly_stats, x='month', y='duration_mean', hue="member_type")
line_monthly_riders = px.line(data_frame=monthly_stats, x='month', y="count_mean", color="member_type",
                               title='Monthly Average Riders by Member Type', width=800, height=450)
line_monthly_riders.show()


In [ ]:
line_weekly_riders = px.line(data_frame=weekly_stats, x='day_of_week', y="count_mean", color="member_type", 
                             title="Average Daily Riders by Member Type", width=800, height=450)
line_weekly_riders.update_layout()
line_weekly_riders.show()

# Conclusion

This analysis demonstrates the statistically significant differences between the rider profiles of Cyclistic's annual members and casual riders. It suggests that focusing on the tending for casual riders to prefer longer rides seasonally and on weekends presents the best path forward. Cleaning, processing, and analyzing a dataset of this size presented many challenges, but allowed for clear patterns to be demonstrated for data-driven decision making, and for future lines of inquiry.